# Results — Parameter Sweep

## Imports

In [ ]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import box
import contextily as ctx
import numpy as np
from scipy.spatial import cKDTree
from scipy.signal import savgol_filter
import matplotlib.dates as mdates
from shapely.strtree import STRtree
from statsmodels.tsa.seasonal import seasonal_decompose
from matplotlib.lines import Line2D
import itertools, os, warnings
warnings.filterwarnings('ignore')

try:
    from dtaidistance import dtw
    from scipy.cluster.hierarchy import linkage, fcluster, dendrogram
    from scipy.spatial.distance import squareform
    from sklearn.preprocessing import StandardScaler
    import seaborn as sns
    from matplotlib.colors import to_hex
    from scipy.stats import pearsonr
except ImportError as e:
    print(f"Erro de Importação: {e}. Instala: dtaidistance, scipy, seaborn, scikit-learn")


## ⚙️ Configuração do Sweep — edita aqui

In [ ]:
# ==============================================================================
# DEFINE AQUI OS VALORES QUE QUERES TESTAR PARA CADA PARÂMETRO
# Basta adicionar mais valores às listas. O sweep corre todas as combinações.
# ==============================================================================

SWEEP = {
    # "TARGET_VAR":      ["dV", "dH"],      # variável a analisar
    "TARGET_VAR":      ["dV"],      # variável a analisar
    "grid_size":       [25, 50],          # tamanho da célula da grelha (metros)
    "radius":          [25, 50],         # raio IDW (metros)
    "power":           [2, 3],                # expoente IDW
    "k":               [3, 5],                # nº de vizinhos IDW
    # "N_CLUSTERS_FIXO": [3, 4],             # nº de clusters
    "N_CLUSTERS_FIXO": [3, 4],             # nº de clusters
    # "DTW_WINDOW_PCT":  [0.01, 0.05],       # janela DTW (fracção da série)
    "DTW_WINDOW_PCT":  [0.01, 0.03],       # janela DTW (fracção da série)
    # "LINKAGE_METHOD":  ["average", "ward"],# método de ligação hierárquico
    "LINKAGE_METHOD":  ["average", "complete"],# método de ligação hierárquico
}

# Pasta raiz onde são guardados os outputs (criada automaticamente)
OUTPUT_ROOT = "sweep_outputs"

# Colocar True para ver prints de progresso de cada run
VERBOSE = True


## Leitura de dados (executada uma vez, fora do loop)

In [ ]:
# ==============================
# 0. Ler COS e definir barragem
# ==============================
COS_PATH = r"C:\projetos\analise_insar_ist\data\COS2023v1-S2-shp\COS2023v1-S2.shp"
try:
    cos = gpd.read_file(COS_PATH).to_crs(epsg=3035)
    barragem = cos[cos['COS23_n4_L'].isin([
        'Infraestruturas de produção de energia hídrica',
        'Equipamentos culturais',
        'Florestas de azinheira',
        'Matos',
        'Pastagens melhoradas',
        'Rede rodoviária',
        'Superfícies silvopastoris de azinheira'
    ])]
except Exception as e:
    print(f"Aviso: COS placeholder. {e}")
    barragem = gpd.GeoDataFrame(
        {'COS23_n4_L': ['Placeholder'],
         'geometry': [box(2792250, 1855050, 2793250, 1855850)]},
        crs="EPSG:3035"
    )

# ==============================
# 1. Ler CSVs e Filtro
# ==============================
asc_file  = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_147_0224_IW2_VV_2019_2023_1/EGMS_L2b_147_0224_IW2_VV_2019_2023_1.csv"
desc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_052_0848_IW2_VV_2019_2023_1/EGMS_L2b_052_0848_IW2_VV_2019_2023_1.csv"

asc_raw  = pd.read_csv(asc_file)
desc_raw = pd.read_csv(desc_file)

norte_min, norte_max = 1855050, 1855850
este_min,  este_max  = 2792250, 2793250

def filter_area(df):
    return df[
        (df['northing'] >= norte_min) & (df['northing'] <= norte_max) &
        (df['easting']  >= este_min)  & (df['easting']  <= este_max)
    ]

asc_raw  = filter_area(asc_raw)
desc_raw = filter_area(desc_raw)

# ==============================
# 2. Melt (uma vez)
# ==============================
def melt_to_long(df):
    disp_cols = df.columns[24:]
    long_df = df.melt(
        id_vars=['easting','northing','incidence_angle','track_angle','latitude','longitude'],
        value_vars=disp_cols, var_name='date', value_name='disp'
    )
    long_df['date'] = pd.to_datetime(long_df['date'], errors='coerce')
    return long_df.dropna(subset=['disp','date'])

asc_long  = melt_to_long(asc_raw)
desc_long = melt_to_long(desc_raw)

common_dates = pd.date_range(
    start=max(asc_long['date'].min(), desc_long['date'].min()),
    end=min(asc_long['date'].max(),   desc_long['date'].max()),
    freq='MS'
)

# ==============================
# 3. Interpolação (uma vez)
# ==============================
def interpolate_ps(df, dates):
    dfs = []
    for (x, y), g in df.groupby(['easting','northing']):
        g = g.sort_values('date')
        interp = np.interp(
            pd.to_datetime(dates).astype(np.int64),
            g['date'].astype(np.int64),
            g['disp']
        )
        dfs.append(pd.DataFrame({
            'easting': x, 'northing': y,
            'latitude': g['latitude'].iloc[0],
            'longitude': g['longitude'].iloc[0],
            'date': dates, 'disp': interp,
            'incidence_angle': g['incidence_angle'].iloc[0],
            'track_angle': g['track_angle'].iloc[0]
        }))
    return pd.concat(dfs, ignore_index=True)

asc_interp_base  = interpolate_ps(asc_long,  common_dates)
desc_interp_base = interpolate_ps(desc_long, common_dates)

# ==============================
# 9. Dados Hidro (uma vez)
# ==============================
win = 13
date_range_hidro = pd.to_datetime(common_dates).normalize()

def process_hidro_variable(df, col_name, is_precip=False):
    try:
        df['data'] = pd.to_datetime(df['data']).dt.normalize()
        df = df.set_index('data')
        s = df[col_name].resample('MS').sum() if is_precip else df[col_name].resample('MS').mean()
        s = s.reindex(date_range_hidro)
        s = s.fillna(0) if is_precip else s.ffill().bfill().fillna(0)
        return s
    except Exception as e:
        print(f"Erro ao processar {col_name}: {e}")
        return pd.Series(0, index=date_range_hidro)

ts_temp  = process_hidro_variable(pd.read_excel("data/alqueva_temp.xlsx"),  'med')
ts_nivel = process_hidro_variable(pd.read_excel("data/alqueva_nivel.xlsx"), 'nivel')
ts_prec  = process_hidro_variable(pd.read_excel("data/prec.xlsx"),          'prec', is_precip=True)

df_temp  = pd.DataFrame({'data': date_range_hidro, 'med':   ts_temp.values})
df_temp['med_smooth']   = savgol_filter(df_temp['med'],   win, 2)

df_nivel = pd.DataFrame({'data': date_range_hidro, 'nivel': ts_nivel.values})
df_nivel['nivel_smooth'] = savgol_filter(df_nivel['nivel'], win, 2)

df_prec  = pd.DataFrame({'data': date_range_hidro, 'prec':  ts_prec.values})
df_prec['prec_acum'] = df_prec['prec'].cumsum()
df_prec['ano_hidrologico'] = df_prec['data'].apply(lambda x: x.year if x.month >= 10 else x.year - 1)
df_prec['prec_acum_anual'] = df_prec.groupby('ano_hidrologico')['prec'].cumsum()

print("✅ Dados carregados. Pronto para o sweep.")


## Funções do pipeline (definidas uma vez)

In [ ]:
# ==============================
# IDW
# ==============================
def idw(source, target, radius, power, k):
    out = []
    for d, src in source.groupby('date'):
        tgt = target[target['date'] == d].copy()
        if src.empty or tgt.empty:
            continue
        tree = cKDTree(list(zip(src['easting'], src['northing'])))
        dist, idx = tree.query(
            list(zip(tgt['easting'], tgt['northing'])),
            k=k, distance_upper_bound=radius
        )
        vals, thetas, alphas = [], [], []
        for d_i, i_i in zip(dist, idx):
            m = np.isfinite(d_i)
            if not np.any(m):
                vals.append(np.nan); thetas.append(np.nan); alphas.append(np.nan)
                continue
            w = 1 / (d_i[m] ** power)
            vals.append(np.sum(w * src.iloc[i_i[m]]['disp'])              / np.sum(w))
            thetas.append(np.sum(w * src.iloc[i_i[m]]['incidence_angle']) / np.sum(w))
            alphas.append(np.sum(w * src.iloc[i_i[m]]['track_angle'])     / np.sum(w))
        tgt['disp_idw']   = vals
        tgt['theta_desc'] = thetas
        tgt['alpha_desc'] = alphas
        out.append(tgt)
    return pd.concat(out, ignore_index=True)

# ==============================
# dV / dH
# ==============================
orb_inc = np.deg2rad(98.6)

def get_comps(row):
    ta, td, beta = (np.deg2rad(row['incidence_angle']),
                    np.deg2rad(row['theta_desc']),
                    row['beta'])
    denom = (np.cos(ta) * np.sin(td) * np.cos(beta) +
             np.cos(td) * np.sin(ta) * np.cos(beta))
    if denom == 0:
        return np.nan, np.nan
    dV = (row['disp_idw'] * np.sin(ta) * np.cos(beta) +
          row['disp']     * np.sin(td) * np.cos(beta)) / denom
    dH = (row['disp_idw'] * np.cos(ta) -
          row['disp']     * np.cos(td)) / denom
    return dV, dH

# ==============================
# Escalonamento visual
# ==============================
def scale_series_visual(data, dV_min, dV_max, scale_factor=0.20, offset_factor=0.30):
    data = np.array(data)
    dmin, dmax = np.nanmin(data), np.nanmax(data)
    dV_span = dV_max - dV_min
    norm    = np.zeros_like(data) if dmax == dmin else (data - dmin) / (dmax - dmin)
    visual  = norm * dV_span * scale_factor + (dV_max - dV_span * offset_factor)
    return visual, dmin, dmax

def create_visual_ticks(d_min_real, d_max_real, dV_min, dV_max,
                        scale_factor=0.20, offset_factor=0.30, num_ticks=5):
    real_ticks = (np.linspace(d_min_real - 0.5, d_max_real + 0.5, num_ticks)
                  if d_max_real == d_min_real
                  else np.linspace(d_min_real, d_max_real, num_ticks))
    dV_span    = dV_max - dV_min
    norm_ticks = (np.linspace(0, 1, num_ticks)
                  if d_max_real == d_min_real
                  else (real_ticks - d_min_real) / (d_max_real - d_min_real))
    visual_ticks = norm_ticks * dV_span * scale_factor + (dV_max - dV_span * offset_factor)
    return visual_ticks, real_ticks

print("✅ Funções definidas.")


## Função `run_one(params)` — o pipeline completo

In [ ]:
def run_one(params):
    """
    Executa o pipeline completo para uma combinação de parâmetros.
    Guarda os outputs (PDFs/PNGs) numa sub-pasta dentro de OUTPUT_ROOT.
    Devolve True se correu sem erros, False caso contrário.
    """
    TARGET_VAR     = params["TARGET_VAR"]
    grid_size      = params["grid_size"]
    radius         = params["radius"]
    power          = params["power"]
    k              = params["k"]
    N_CLUSTERS_FIXO= params["N_CLUSTERS_FIXO"]
    DTW_WINDOW_PCT = params["DTW_WINDOW_PCT"]
    LINKAGE_METHOD = params["LINKAGE_METHOD"]

    # --- pasta de output ---
    run_name = (f"{TARGET_VAR}_gs{grid_size}_r{radius}_p{power}_k{k}"
                f"_nc{N_CLUSTERS_FIXO}_dtw{DTW_WINDOW_PCT}_lnk{LINKAGE_METHOD}")
    run_dir  = os.path.join(OUTPUT_ROOT, run_name)
    os.makedirs(run_dir, exist_ok=True)

    def save(fname):
        return os.path.join(run_dir, fname)

    if VERBOSE:
        print(f"  → {run_name}")

    try:
        # ── 4. IDW ──────────────────────────────────────────────────────────
        asc_i = asc_interp_base.copy()
        desc_i = desc_interp_base.copy()
        asc_i = idw(desc_i, asc_i, radius=radius, power=power, k=k)
        asc_i = asc_i.dropna(subset=["disp_idw"])

        # ── 5. dV / dH ──────────────────────────────────────────────────────
        asc_i["beta"] = np.arcsin(
            np.cos(orb_inc) * np.cos(np.deg2rad(asc_i["latitude"]))
        )
        asc_i[["dV", "dH"]] = asc_i.apply(
            lambda x: pd.Series(get_comps(x)), axis=1
        )
        asc_i = asc_i.dropna(subset=["dV", "dH"])

        # ── 6. Grelha ────────────────────────────────────────────────────────
        xe = np.arange(asc_i["easting"].min(),  asc_i["easting"].max()  + grid_size, grid_size)
        ye = np.arange(asc_i["northing"].min(), asc_i["northing"].max() + grid_size, grid_size)
        xs, ys = xe - grid_size / 2, ye - grid_size / 2

        asc_i["cx"] = pd.cut(asc_i["easting"],  bins=xs, labels=False)
        asc_i["cy"] = pd.cut(asc_i["northing"], bins=ys, labels=False)
        asc_i = asc_i.dropna(subset=["cx", "cy"])
        asc_i["cell_id"] = (asc_i["cx"].astype(int).astype(str) + "_" +
                            asc_i["cy"].astype(int).astype(str))

        grid_data = [
            {"cell_id": f"{ix}_{iy}",
             "geometry": box(xs[ix], ys[iy], xs[ix+1], ys[iy+1])}
            for ix in range(len(xs)-1)
            for iy in range(len(ys)-1)
        ]
        grid = gpd.GeoDataFrame(grid_data, crs="EPSG:3035")

        agg = asc_i.groupby(["cell_id", "date"]).agg(
            dV=(TARGET_VAR, "mean")
        ).reset_index()

        # ── 7. Recorte ───────────────────────────────────────────────────────
        points_gdf   = gpd.GeoDataFrame(
            asc_i,
            geometry=gpd.points_from_xy(asc_i["easting"], asc_i["northing"]),
            crs="EPSG:3035"
        ).to_crs(epsg=3857)
        grid_3857     = grid.to_crs(epsg=3857)
        barragem_3857 = barragem.to_crs(epsg=3857)
        grid_recort   = gpd.overlay(grid_3857, barragem_3857, how="intersection")

        poly  = grid_recort.geometry.values
        ids   = grid_recort["cell_id"].values
        stree = STRtree(poly)
        valid = set()
        for pt in points_gdf.geometry:
            for sidx in stree.query(pt):
                if poly[sidx].contains(pt):
                    valid.add(ids[sidx])

        grid_barragem = grid_recort[grid_recort["cell_id"].isin(valid)]
        agg           = agg[agg["cell_id"].isin(valid)]

        # ── 8. Clustering DTW ────────────────────────────────────────────────
        if VERBOSE:
            print(f"     DTW (window={DTW_WINDOW_PCT})...")

        agg_pivot = agg.pivot(index="cell_id", columns="date", values="dV")
        agg_pivot = agg_pivot.interpolate(axis=1, limit_direction="both").dropna()

        cell_ids = agg_pivot.index.tolist()
        X        = agg_pivot.values.astype(float)
        n        = X.shape[0]

        if n < 2:
            print(f"  ⚠️  Células insuficientes ({n}). A saltar.")
            return False

        X_scaled     = X
        window_dtw   = max(1, int(DTW_WINDOW_PCT * X_scaled.shape[1]))
        dist_matrix  = dtw.distance_matrix_fast(X_scaled, window=window_dtw)
        condensed    = squareform(dist_matrix)
        Z            = linkage(condensed, method=LINKAGE_METHOD)

        cluster_labels = fcluster(Z, t=N_CLUSTERS_FIXO, criterion="maxclust")
        u_labels       = np.unique(cluster_labels)
        map_l          = {old: new for new, old in enumerate(u_labels)}
        cluster_labels = np.array([map_l[x] for x in cluster_labels])
        num_clusters   = len(u_labels)

        cluster_df    = pd.DataFrame({"cell_id": cell_ids, "cluster": cluster_labels})
        palette       = (sns.color_palette("Set2",  num_clusters) if num_clusters <= 8
                         else sns.color_palette("tab20", num_clusters))
        cluster_colors = {cl: to_hex(c) for cl, c in zip(range(num_clusters), palette)}
        grid_sel      = grid_barragem.merge(cluster_df, on="cell_id", how="inner")

        # ── FIGURA DENDROGRAMA ───────────────────────────────────────────────
        try:
            distancia_visual = Z[-num_clusters + 1, 2] if num_clusters > 1 else 0
        except Exception:
            distancia_visual = 0

        fig_dend, ax_dend = plt.subplots(figsize=(12, 5))
        dendrogram(Z, leaf_rotation=90, leaf_font_size=8,
                   color_threshold=distancia_visual, no_labels=True, ax=ax_dend)
        ax_dend.axhline(y=distancia_visual, c="k", ls="--", lw=1.2,
                        label=f"Corte K={num_clusters}")
        ax_dend.set_title(f"Dendrograma (K={num_clusters}, {LINKAGE_METHOD})")
        ax_dend.set_xlabel("Células"); ax_dend.set_ylabel("Distância")
        ax_dend.legend(); fig_dend.tight_layout()
        fig_dend.savefig(save("dendrograma.pdf"), format="pdf", bbox_inches="tight")
        plt.close(fig_dend)

        # ── FIGURA 1: MAPA ───────────────────────────────────────────────────
        clusters_present = sorted(cluster_df["cluster"].unique())
        k_plot = len(clusters_present)

        fig_map = plt.figure(figsize=(12, 12))
        ax_map  = fig_map.add_subplot(1, 1, 1)
        grid.to_crs(epsg=3857).boundary.plot(ax=ax_map, color="white", lw=0.5, alpha=0.5)
        grid_sel.boundary.plot(ax=ax_map, color="black", lw=1)

        for i in clusters_present:
            sub = grid_sel[grid_sel["cluster"] == i]
            sub.plot(ax=ax_map, color=cluster_colors[i], alpha=0.4)
            cents = sub.copy(); cents.geometry = cents.geometry.centroid
            cents.plot(ax=ax_map, marker="o", color="white",
                       edgecolor="black", markersize=30, zorder=5)

        ctx.add_basemap(ax_map, source=ctx.providers.Esri.WorldImagery)
        ax_map.set_axis_off()
        hdl = [plt.Rectangle((0,0),1,1, fc=cluster_colors[i], alpha=0.4) for i in clusters_present]
        lbl = [f"Cluster {i+1} ({len(grid_sel[grid_sel['cluster']==i])} cel)" for i in clusters_present]
        hdl.append(Line2D([0],[0], marker="o", color="w", markerfacecolor="white",
                           markeredgecolor="black", markersize=8, linestyle="None"))
        lbl.append("Centróides")
        ax_map.legend(hdl, lbl, loc="upper left", fontsize=10,
                      title=f"Clusters DTW (K={k_plot})")
        ax_map.set_title(f"Mapa de Clusters ({TARGET_VAR})", fontsize=15)
        fig_map.savefig(save("mapa.pdf"), format="pdf")
        plt.close(fig_map)

        # ── FIGURA 2: SÉRIES TEMPORAIS ───────────────────────────────────────
        n_rows_series = 5
        fig_series = plt.figure(figsize=(5.5 * k_plot, 3.0 * n_rows_series))
        gs_series  = fig_series.add_gridspec(
            n_rows_series, k_plot, height_ratios=[3.0] * n_rows_series
        )

        dV_min    = agg["dV"].min()
        dV_max    = agg["dV"].max()
        dV_margin = (dV_max - dV_min) * 0.1
        dV_ylim   = (dV_min - dV_margin, dV_max + dV_margin)

        temp_visual,  _, _ = scale_series_visual(df_temp["med_smooth"],   dV_min, dV_max)
        nivel_visual, _, _ = scale_series_visual(df_nivel["nivel_smooth"], dV_min, dV_max)

        for idx, cluster_id in enumerate(clusters_present):
            cluster_cells = cluster_df[cluster_df["cluster"] == cluster_id]["cell_id"].tolist()
            cluster_data  = agg_pivot.loc[cluster_cells]
            cluster_mean  = cluster_data.mean(axis=0)
            cluster_color = cluster_colors[cluster_id]
            n_cells       = len(cluster_cells)

            def plot_row(row_idx, ext_data_visual, ext_df, ext_col,
                         ext_label, ext_color,
                         plot_as_bar=False, bar_color=None,
                         plot_real_scale_line=False,
                         combine_annual_prec=False,
                         integer_ticks=False, show_background_bars=False):

                ax  = fig_series.add_subplot(gs_series[row_idx, idx])
                for cid in cluster_cells:
                    ax.plot(cluster_data.columns, cluster_data.loc[cid],
                            color="lightgray", alpha=0.7, linewidth=1.0)
                ax.plot(cluster_data.columns, cluster_mean,
                        color=cluster_color, linewidth=1.0,
                        label=f"Média Cluster {cluster_id+1}")

                ax2    = ax.twinx()
                lines2 = []

                if combine_annual_prec:
                    ax2.bar(df_prec["data"], df_prec["prec"], width=15,
                            color="teal", alpha=0.3)
                    for ano in df_prec["ano_hidrologico"].unique():
                        grupo = df_prec[df_prec["ano_hidrologico"] == ano]
                        ax2.plot(grupo["data"], grupo["prec_acum_anual"],
                                 color="teal", linewidth=1.0, alpha=0.9)
                    ax2.set_ylim(0, df_prec["prec_acum_anual"].max() * 1.1)
                    ax2.set_ylabel("Prec. acumulada anual", color="teal")
                    ax2.tick_params(axis="y", colors="teal")
                    lines2 = [
                        plt.Rectangle((0,0),1,1, fc="teal", alpha=0.3,
                                      label="Precipitação mensal (mm)"),
                        plt.Line2D([0],[0], color="teal", linewidth=1.0,
                                   label="Prec. acumulada anual (mm)")
                    ]
                elif plot_as_bar:
                    ax2.bar(ext_df["data"], ext_df[ext_col], width=15,
                            color=bar_color or ext_color, alpha=0.5)
                    ax2.set_ylim(0, ext_df[ext_col].max() * 1.1)
                    ax2.set_ylabel(ext_label, color="teal")
                    ax2.tick_params(axis="y", colors="teal")
                    lines2 = [plt.Rectangle((0,0),1,1, fc=bar_color or ext_color,
                                            alpha=0.5, label=ext_label)]
                elif plot_real_scale_line:
                    if show_background_bars:
                        ax2.bar(df_prec["data"], df_prec["prec"], width=15,
                                color="teal", alpha=0.3)
                    ax2.plot(ext_df["data"], ext_df[ext_col],
                             color="teal", linewidth=1.0, alpha=0.85)
                    ax2.set_ylim(0, ext_df[ext_col].max() * 1.1)
                    ax2.set_ylabel(ext_label, color="teal")
                    ax2.tick_params(axis="y", colors="teal")
                    lines2 = [
                        plt.Rectangle((0,0),1,1, fc="teal", alpha=0.3,
                                      label="Precipitação mensal (mm)"),
                        plt.Line2D([0],[0], color="teal", linewidth=1.0, label=ext_label)
                    ]
                else:
                    ax2.plot(ext_df["data"], ext_data_visual,
                             color=ext_color, linewidth=1.0, alpha=0.85)
                    vt, rt = create_visual_ticks(
                        ext_df[ext_col].min(), ext_df[ext_col].max(), dV_min, dV_max
                    )
                    ax2.set_ylim(dV_ylim); ax2.set_yticks(vt)
                    ax2.set_yticklabels([f"{t:.0f}" if integer_ticks else f"{t:.1f}"
                                         for t in rt])
                    lc = "black" if "Temperatura" in ext_label else ext_color
                    ax2.set_ylabel(ext_label, color=lc)
                    ax2.tick_params(axis="y", colors=lc)
                    lines2 = [ax2.lines[-1]]

                ax.set_ylim(dV_ylim)
                ax.set_title(f"Cluster {cluster_id+1} ({n_cells} células)", fontsize=12)
                if idx == 0:
                    ax.set_ylabel("dV (mm)")
                else:
                    ax.set_yticklabels([])
                if idx != k_plot - 1:
                    ax2.set_yticklabels([])
                ax.tick_params(left=(idx == 0))
                ax2.tick_params(right=(idx == k_plot - 1))
                if row_idx == n_rows_series - 1:
                    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
                else:
                    ax.set_xticklabels([])
                if idx == 0 or idx == k_plot - 1:
                    l1, lb1 = ax.get_legend_handles_labels()
                    h2, lb2 = ax2.get_legend_handles_labels()
                    ax.legend(l1 + h2, lb1 + lb2, fontsize=8, loc="best",
                              framealpha=1.0, facecolor="white",
                              edgecolor="lightgray").set_zorder(100)

            plot_row(0, temp_visual,  df_temp,  "med_smooth",
                     "Temperatura média (°C)", "black", integer_ticks=True)
            plot_row(1, nivel_visual, df_nivel, "nivel_smooth",
                     "Nível da albufeira (m)", "navy",  integer_ticks=True)
            plot_row(2, None, df_prec, "prec",
                     "Precipitação mensal (mm)", "teal",
                     plot_as_bar=True, bar_color="teal")
            plot_row(3, None, df_prec, "prec_acum",
                     "Prec. acumulada total (mm)", "teal",
                     plot_real_scale_line=True, show_background_bars=True)
            plot_row(4, None, df_prec, "prec_acum_anual",
                     "Prec. acumulada anual", "teal",
                     combine_annual_prec=True)

        fig_series.tight_layout()
        fig_series.savefig(save("clusters.pdf"), format="pdf")
        plt.close(fig_series)

        # # ── FIGURA 3: DECOMPOSIÇÃO SAZONAL ───────────────────────────────────
        # fig_decomp, axes = plt.subplots(
        #     4, k_plot, figsize=(6 * max(1, k_plot), 7), sharex=False
        # )
        # if k_plot == 1:
        #     axes = axes.reshape(4, 1)

        # for col, cluster_id in enumerate(clusters_present):
        #     cluster_cells = cluster_df[cluster_df["cluster"] == cluster_id]["cell_id"].tolist()
        #     mean_series   = agg_pivot.loc[cluster_cells].mean(axis=0)
        #     decomp        = seasonal_decompose(mean_series, model="additive", period=12)
        #     c_color       = cluster_colors.get(cluster_id, "black")

        #     for row, (label, series) in enumerate([
        #         ("Observed",  decomp.observed),
        #         ("Trend",     decomp.trend),
        #         ("Seasonal",  decomp.seasonal),
        #         ("Residual",  decomp.resid),
        #     ]):
        #         ax = axes[row, col]
        #         ax.plot(series.index, series.values, color=c_color, linewidth=1.0)
        #         for sp in ax.spines.values():
        #             sp.set_visible(True); sp.set_linewidth(0.8); sp.set_color("black")
        #         if label != "Observed":
        #             ymin, ymax = np.nanmin(series), np.nanmax(series)
        #             if np.isnan(ymin) or np.isnan(ymax): ymin, ymax = -1, 1
        #             margin = (ymax - ymin) * 0.10
        #             ax.set_ylim(ymin - margin, ymax + margin)
        #         if row == 0:
        #             ax.set_title(f"Seasonal Decompose — Cluster {cluster_id+1}", fontsize=12)
        #         if col == 0:
        #             ax.set_ylabel(label, fontsize=10)
        #         else:
        #             ax.set_yticks([])
        #         if row == 3:
        #             ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
        #             ax.xaxis.set_major_locator(mdates.YearLocator())
        #             ax.tick_params(axis="x", labelrotation=0, labelsize=10)
        #         else:
        #             ax.set_xticks([]); ax.set_xticklabels([])

        # fig_decomp.tight_layout()
        # fig_decomp.savefig(save("stl.pdf"), format="pdf")
        # plt.close(fig_decomp)
        
        # ── FIGURA 3: DECOMPOSIÇÃO SAZONAL (CORRIGIDA) ───────────────────────────────────
        # O segredo está no sharey='row' para permitir comparação entre clusters
        fig_decomp, axes = plt.subplots(
            4, k_plot, figsize=(6 * max(1, k_plot), 8), sharex=False, sharey='row'
        )

        # Garantir que axes seja sempre 2D mesmo com 1 cluster
        if k_plot == 1:
            axes = axes.reshape(4, 1)

        for col, cluster_id in enumerate(clusters_present):
            cluster_cells = cluster_df[cluster_df["cluster"] == cluster_id]["cell_id"].tolist()
            # Usamos agg_pivot que já contém os valores reais de dV
            mean_series   = agg_pivot.loc[cluster_cells].mean(axis=0)
            
            # Decomposição (Additive é o correto para InSAR dV)
            decomp = seasonal_decompose(mean_series, model="additive", period=12)
            c_color = cluster_colors.get(cluster_id, "black")

            components = [
                ("Observed", decomp.observed),
                ("Trend",    decomp.trend),
                ("Seasonal", decomp.seasonal),
                ("Residual", decomp.resid),
            ]

            for row, (label, series) in enumerate(components):
                ax = axes[row, col]
                
                # Plot da série
                ax.plot(series.index, series.values, color=c_color, linewidth=1.2)
                
                # Estilização das molduras
                for sp in ax.spines.values():
                    sp.set_visible(True)
                    sp.set_linewidth(0.8)
                    sp.set_color("black")
                    
                # Títulos e Labels
                if row == 0:
                    ax.set_title(f"Cluster {cluster_id+1}\n({len(cluster_cells)} células)", fontsize=11, fontweight='bold')
                
                if col == 0:
                    ax.set_ylabel(label, fontsize=10, fontweight='bold')
                else:
                    # sharey='row' já cuida da escala, mas limpamos os labels para estética
                    ax.tick_params(axis='y', labelleft=False)

                # Formatação do eixo X (Datas) apenas na última linha
                if row == 3:
                    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
                    ax.xaxis.set_major_locator(mdates.YearLocator())
                    ax.tick_params(axis="x", labelrotation=0, labelsize=9)
                else:
                    ax.set_xticks([])
                    ax.set_xticklabels([])

        # Ajuste de layout e salvamento
        fig_decomp.tight_layout()
        fig_decomp.savefig(save("stl.pdf"), format="pdf")
        plt.close(fig_decomp)

        # ── FIGURA 4: PAINEL CORRELAÇÃO ──────────────────────────────────────
        LINE_WIDTH = 0.4; LINE_COLOR = "#333333"; MY_CMAP = plt.cm.RdBu
        sns.set_context("paper", font_scale=0.8)
        plt.rcParams["axes.linewidth"] = LINE_WIDTH

        def desenhar_matriz_cluster(ax_subgrid, df_cl, cluster_id):
            cols   = df_cl.columns.tolist()
            n_vars = len(cols)
            sub_gs = ax_subgrid.get_subplotspec().subgridspec(n_vars, n_vars, hspace=0, wspace=0)
            axes_list = []
            for i in range(n_vars):
                row_axes = []
                for j in range(n_vars):
                    ax = plt.subplot(sub_gs[i, j])
                    xd, yd  = df_cl[cols[j]], df_cl[cols[i]]
                    xmn, xmx = xd.min(), xd.max()
                    ymn, ymx = yd.min(), yd.max()
                    xr = (xmx - xmn) if xmx != xmn else 1
                    yr = (ymx - ymn) if ymx != ymn else 1
                    if i == j:
                        sns.histplot(xd, kde=True, color="#2c3e50", alpha=0.2,
                                     edgecolor="white", linewidth=0.1, ax=ax)
                        ax.annotate(cols[i], xy=(0.05, 0.90), xycoords="axes fraction",
                                    fontsize=6, color="red")
                        ax.set_xlim(xmn - 0.3*xr, xmx + 0.3*xr)
                    elif i > j:
                        ax.scatter(xd, yd, s=1, color="black", alpha=1.0, linewidths=0)
                        sns.regplot(x=xd, y=yd, ax=ax, ci=None, scatter=False,
                                    line_kws={"color": "red", "linewidth": 0.5})
                        ax.set_xlim(xmn - 0.3*xr, xmx + 0.3*xr)
                        ax.set_ylim(ymn - 0.3*yr, ymx + 0.3*yr)
                    else:
                        r, _ = pearsonr(xd, yd)
                        ax.set_facecolor((*MY_CMAP((r + 1) / 2)[:3], 0.3))
                        ax.annotate(f"{r:.2f}", xy=(0.5, 0.5), xycoords="axes fraction",
                                    ha="center", va="center", fontsize=7)
                    ax.set_xlabel(""); ax.set_ylabel("")
                    ax.tick_params(labelsize=0, direction="in", pad=0,
                                   width=LINE_WIDTH, length=0)
                    if j == 0: ax.tick_params(axis="y", left=True, labelsize=5, length=2)
                    if i == n_vars-1: ax.tick_params(axis="x", bottom=True, labelsize=5, length=2)
                    for edge in ax.spines: ax.spines[edge].set_linewidth(LINE_WIDTH)
                    row_axes.append(ax)
                axes_list.append(row_axes)
            axes_list[0][int(n_vars/2)].annotate(
                f"Cluster {cluster_id+1}", xy=(0.5, 1.25), xycoords="axes fraction",
                ha="center", fontweight="bold", fontsize=9
            )

        n_cl = len(clusters_present)
        fig_corr = plt.figure(figsize=(4 * n_cl + 0.5, 6.5))
        master_gs = fig_corr.add_gridspec(
            1, n_cl, wspace=0.4, left=0.08, right=0.92, bottom=0.38, top=0.88
        )
        for idx2, cluster_id in enumerate(clusters_present):
            c_cells = cluster_df[cluster_df["cluster"] == cluster_id]["cell_id"]
            dV_mean = agg_pivot.loc[c_cells].mean()
            df_cl   = pd.DataFrame({
                "dV":   dV_mean,
                "Temp": df_temp.set_index("data")["med_smooth"].reindex(dV_mean.index),
                "Niv":  df_nivel.set_index("data")["nivel_smooth"].reindex(dV_mean.index),
                "Prec": df_prec.set_index("data")["prec"].reindex(dV_mean.index),
                "P_Ac": df_prec.set_index("data")["prec_acum"].reindex(dV_mean.index),
                "P_An": df_prec.set_index("data")["prec_acum_anual"].reindex(dV_mean.index),
            }).dropna()
            ax_ph = fig_corr.add_subplot(master_gs[idx2])
            ax_ph.axis("off")
            desenhar_matriz_cluster(ax_ph, df_cl, cluster_id)

        cax  = fig_corr.add_axes([0.35, 0.28, 0.30, 0.02])
        sm   = plt.cm.ScalarMappable(cmap=MY_CMAP, norm=plt.Normalize(-1, 1))
        cbar = fig_corr.colorbar(sm, cax=cax, orientation="horizontal")
        cbar.set_label("Correlation coefficient (r)", fontsize=8, labelpad=5)
        cbar.ax.tick_params(labelsize=7, width=LINE_WIDTH)
        cbar.outline.set_linewidth(LINE_WIDTH)

        ax_table = fig_corr.add_axes([0.1, 0.05, 0.8, 0.12])
        ax_table.axis("off")
        legend_data = [
            ["dV: Vertical Displacement", "Temp: Temperature",      "Niv: Reservoir Level"],
            ["Prec: Daily Precipitation", "P_Ac: Accumulated Prec.","P_An: Annual Acc. Prec."]
        ]
        table = ax_table.table(cellText=legend_data, loc="center", cellLoc="left")
        table.auto_set_font_size(False); table.set_fontsize(7.5); table.scale(1, 1.4)
        for cell in table.get_celld().values():
            cell.set_linewidth(LINE_WIDTH); cell.set_edgecolor(LINE_COLOR)


        fig_corr.tight_layout()
        fig_corr.savefig(save("correlacao.pdf"), format="pdf")
        plt.close(fig_corr)

        if VERBOSE:
            print(f"     ✅ Guardado em: {run_dir}")
        return True

    except Exception as e:
        import traceback
        print(f"  ❌ ERRO: {e}")
        traceback.print_exc()
        return False


print("✅ run_one() definida.")


## ▶️ Correr o Sweep

In [ ]:
# Gera todas as combinações
keys   = list(SWEEP.keys())
values = list(SWEEP.values())
combos = [dict(zip(keys, v)) for v in itertools.product(*values)]

total   = len(combos)
success = 0
failed  = []

print(f"🚀 A iniciar sweep: {total} combinações")
print(f"   Outputs em: {os.path.abspath(OUTPUT_ROOT)}\n")

for i, params in enumerate(combos, 1):
    print(f"[{i:3d}/{total}]", end=" ")
    ok = run_one(params)
    if ok:
        success += 1
    else:
        failed.append(params)

print(f"\n{'='*60}")
print(f"✅ Concluído: {success}/{total} runs com sucesso")
if failed:
    print(f"❌ Falharam {len(failed)} runs:")
    for p in failed:
        print("   ", p)


In [ ]:
# ==============================================================================
# 🏆 ANÁLISE FINAL: RANKING VISUAL E ESCOLHA DO MELHOR CENÁRIO (CORRIGIDO)
# ==============================================================================
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from scipy.stats import pearsonr
import os

print("📊 Gerando relatório visual de performance...")

results_list = []

for params in combos:
    try:
        # --- Re-processamento simplificado para score ---
        asc_i = idw(desc_interp_base.copy(), asc_interp_base.copy(), 
                    radius=params['radius'], power=params['power'], k=params['k']).dropna(subset=["disp_idw"])
        asc_i["beta"] = np.arcsin(np.cos(orb_inc) * np.cos(np.deg2rad(asc_i["latitude"])))
        asc_i[["dV", "dH"]] = asc_i.apply(lambda x: pd.Series(get_comps(x)), axis=1)
        asc_i = asc_i.dropna(subset=["dV", "dH"])
        
        G = params['grid_size']
        xe = np.arange(asc_i["easting"].min(), asc_i["easting"].max() + G, G)
        ye = np.arange(asc_i["northing"].min(), asc_i["northing"].max() + G, G)
        asc_i["cell_id"] = (pd.cut(asc_i["easting"], bins=xe-G/2, labels=False).astype(str) + "_" +
                            pd.cut(asc_i["northing"], bins=ye-G/2, labels=False).astype(str))
        
        agg_sub = asc_i.groupby(["cell_id", "date"]).agg(dV=(params['TARGET_VAR'], "mean")).reset_index()
        pivot = agg_sub.pivot(index="cell_id", columns="date", values="dV").interpolate(axis=1).dropna()
        
        X = pivot.values.astype(float)
        win = max(1, int(params['DTW_WINDOW_PCT'] * X.shape[1]))
        Z = linkage(squareform(dtw.distance_matrix_fast(X, window=win)), method=params['LINKAGE_METHOD'])
        labels = fcluster(Z, t=params['N_CLUSTERS_FIXO'], criterion="maxclust")
        
        # --- Cálculo da Correlação Média (4 Variáveis: Temp, Niv, P_Acum, P_Anual) ---
        cluster_scores = []
        for cl_id in np.unique(labels):
            dV_mean = pivot.loc[pivot.index[labels == cl_id]].mean()
            df_m = pd.DataFrame({
                "dV": dV_mean,
                "Temp": df_temp.set_index("data")["med_smooth"].reindex(dV_mean.index),
                "Niv":  df_nivel.set_index("data")["nivel_smooth"].reindex(dV_mean.index),
                "P_Ac": df_prec.set_index("data")["prec_acum"].reindex(dV_mean.index),
                "P_An": df_prec.set_index("data")["prec_acum_anual"].reindex(dV_mean.index)
            }).dropna()
            
            c_temp, _ = pearsonr(df_m["dV"], df_m["Temp"])
            c_niv, _  = pearsonr(df_m["dV"], df_m["Niv"])
            c_pac, _  = pearsonr(df_m["dV"], df_m["P_Ac"])
            c_pan, _  = pearsonr(df_m["dV"], df_m["P_An"])
            
            # Média dos valores absolutos das 4 correlações
            cluster_scores.append(np.mean([abs(c_temp), abs(c_niv), abs(c_pac), abs(c_pan)]))
        
        score_final = np.mean(cluster_scores)
        
        results_list.append({
            "ID": f"Grid{G}_R{params['radius']}_{params['LINKAGE_METHOD']}_K{params['N_CLUSTERS_FIXO']}",
            "Grelha": f"{G}m",
            "Raio": f"{params['radius']}m",
            "DTW": params['DTW_WINDOW_PCT'],
            "Linkage": params['LINKAGE_METHOD'],
            "Clusters": params['N_CLUSTERS_FIXO'],
            "Score": round(score_final, 4)
        })
    except Exception:
        continue

# 2. Ranking
df_rank = pd.DataFrame(results_list).sort_values(by="Score", ascending=False).reset_index(drop=True)

# ── GERAÇÃO DA FIGURA FINAL ──────────────────────────────────────────────────
fig = plt.figure(figsize=(14, 12))
gs = fig.add_gridspec(2, 1, height_ratios=[1, 1.2], hspace=0.3)

# PARTE SUPERIOR: Gráfico de Barras
ax_bar = fig.add_subplot(gs[0])
top_10 = df_rank.head(10).copy()
colors = ['#27ae60' if i == 0 else '#34495e' for i in range(len(top_10))]
bars = ax_bar.bar(top_10["ID"], top_10["Score"], color=colors, alpha=0.8)
ax_bar.set_title("Top 10 Cenários - Performance de Correlação (Temp+Niv+P.Ac+P.An)", fontsize=14, fontweight='bold')
ax_bar.set_ylabel("Score Médio (|r|)")
ax_bar.set_ylim(0, 1.1)
plt.setp(ax_bar.get_xticklabels(), rotation=25, ha="right")

for bar in bars:
    yval = bar.get_height()
    ax_bar.text(bar.get_x() + bar.get_width()/2, yval + 0.01, f"{yval:.4f}", ha='center', va='bottom', fontsize=9)

# PARTE INFERIOR: Tabela
ax_tbl = fig.add_subplot(gs[1])
ax_tbl.axis('off')
top_15_table = df_rank.head(15).drop(columns=["ID"])

table = ax_tbl.table(
    cellText=top_15_table.values,
    colLabels=top_15_table.columns,
    loc='center',
    cellLoc='center',
    colColours=["#f2f2f2"] * len(top_15_table.columns)
)
table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(1, 2)

# Destacar Vencedor
for j in range(len(top_15_table.columns)):
    table[(1, j)].set_facecolor("#d4efdf")

# Conclusão
melhor_id = df_rank.iloc[0]["ID"]
fig.text(0.5, 0.02, f"🏆 MELHOR CENÁRIO SELECIONADO: {melhor_id}", 
         ha="center", fontsize=15, fontweight='bold', color='#27ae60', 
         bbox=dict(facecolor='white', alpha=0.9, edgecolor='#27ae60', boxstyle='round,pad=0.5'))

# Salvar
output_path = os.path.join(OUTPUT_ROOT, "Relatorio_Final_Sweep.pdf")
plt.savefig(output_path, bbox_inches='tight')
plt.show()

print(f"\n✅ Relatório salvo em: {output_path}")